# Name game: predicting author gender with the NYSIIS phonetic algorithm

**Status:** ✅ Runs end to end — **the method is weak on its own terms**, and the notebook says so in its own output

**Demonstrates:** String algorithms · fuzzy matching across datasets. The phonetic join is lossy (`beach`/`bitch` collide), so the trend it finds is a property of the join, not of the authors.

---

> Part of a portfolio of applied ML and statistics work. Every notebook
> here is executed end to end; the bug record is in `BUGFIXES.md`.



Grey and Gray. Colour and Color. Words like these have been the cause of many heated arguments between Brits and Americans. 

Accents (and jokes) aside, there are many words that are pronounced the same way but have different spellings. While it is easy for us to realize their equivalence, basic programming commands will fail to equate such two strings. 
More extreme than word spellings are names because people have more flexibility in choosing to spell a name in a certain way. To some extent, tradition sometimes governs the way a name is spelled, which limits the number of variations of any given English name. But if we consider global names and their associated English spellings, you can only imagine how many ways they can be spelled out. 


In [ ]:
import sys
from pathlib import Path

# The compatibility shim lives alongside the tested package in src/. Putting
# that directory on sys.path here avoids requiring an install to run a
# notebook.
_SRC = Path.cwd().parents[1] / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

"""NYSIIS phonetic encoding, with a fallback for the un-installable `fuzzy` package.

The original notebook called `fuzzy.nysiis()`. The `fuzzy` distribution on PyPI
ships as a C extension and needs a working compiler plus Cython at install
time, so on many machines `pip install fuzzy` simply fails. That made the
notebook unrunnable for anyone who had not already installed it.

This module resolves the function once, preferring the original and falling
back to `abydos`, which is pure Python and installs everywhere:

    >>> from nysiis_compat import nysiis
    >>> nysiis("Pfister")
    'FASTAR'

**A warning about the fallback.** NYSIIS has several published
implementations and they do not agree on every name. `abydos` agrees with the
reference on cases like Macintosh/McIntosh and Pfister/Fister, but differs on
others (it gives Smith -> SNAT where some references give SMAD). If
`fuzzy` is available it is used and this concern disappears. Which is
*available* cannot change the notebook's conclusions either way - see the
caveat at the end of the notebook, which is the real limitation of this
approach.
"""

from __future__ import annotations

__all__ = ["nysiis", "BACKEND"]

try:  # the original, preferred
    from fuzzy import nysiis

    BACKEND = "fuzzy"
except ImportError:  # pragma: no cover - depends on the environment
    try:
        from abydos.phonetic import NYSIIS

        _encoder = NYSIIS()

        def nysiis(name: str) -> str:
            """Compute the NYSIIS phonetic key for a name.

            Args:
                name: Any string. Non-alphabetic characters are discarded.

            Returns:
                str: The uppercase phonetic key.
            """
            return _encoder.encode(name)

        BACKEND = "abydos"
    except ImportError:  # pragma: no cover
        nysiis = None
        BACKEND = None


if BACKEND is None:
    raise ImportError(
        "Neither `fuzzy` nor `abydos` is installed. Install either with:\n"
        "    pip install fuzzy     # original, needs a C compiler\n"
        "    pip install abydos    # pure Python fallback"
    )
print("NYSIIS backend:", BACKEND)


In [ ]:
# BUGFIX: the original did `import fuzzy` unconditionally. The `fuzzy`
# distribution needs a C extension and a compiler, so this cell raised
# ImportError on most machines and the notebook could not run at all. The
# import now goes through nysiis_compat, which falls back to a pure-Python
# implementation.
from nysiis_compat import nysiis

# Exploring the output of the encoder
print("Harel ->", nysiis("Harel"))

# Testing equivalence of similar-sounding words
print("colour == color :", nysiis("colour") == nysiis("color"))
print("beach  == bitch  :", nysiis("beach") == nysiis("bitch"),
      "  <- the collision that makes this approach unreliable")


## 2. Authoring the authors
<p>The New York Times puts out a weekly list of best-selling books from different genres, and which has been published since the 1930’s.  I’ll focus on Children’s Picture Books, and analyze the gender distribution of authors to see if there have been changes over time. I will begin by reading in the data on the best selling authors from 2008 to 2017.</p>

In [2]:
# Importing the pandas module
import pandas as pd

# Reading in ../../data/nlp/nytkids_yearly.csv, which is semicolon delimited.
author_df = pd.read_csv('../../data/nlp/nytkids_yearly.csv', sep = ';')

# Looping through author_df['Author'] to extract the authors first names
first_name = []
for name in author_df['Author']:
    n= list(name.split())
    first_name.append(n[0])
    

# Adding first_name as a column to author_df
author_df['first_name']= first_name

# Checking out the first few rows of author_df
author_df.head()

,Year,Book Title,Author,Besteller this year,first_name
0,2017,DRAGONS LOVE TACOS,Adam Rubin,49,Adam
1,2017,THE WONDERFUL THINGS YOU WILL BE,Emily Winfield Martin,48,Emily
2,2017,THE DAY THE CRAYONS QUIT,Drew Daywalt,44,Drew
3,2017,"ROSIE REVERE, ENGINEER",Andrea Beaty,38,Andrea
4,2017,"ADA TWIST, SCIENTIST",Andrea Beaty,28,Andrea


## 3.using phonic
I will create a new column or list that contains the phonetic equivalent of every first name that </p>
As a rule of thumb, the number of unique nysiis first names should be less than or equal to the number of actual first names.</p>

In [3]:
# Importing numpy
import numpy as np


# Looping through author's first names to create the nysiis (fuzzy) equivalent
# BUGFIX: was `fuzzy.nysiis(...)`, which crashes whenever the
# `fuzzy` package could not be installed. Goes through the shim now.
nysiis_name = [nysiis(name) for name in first_name]


# Adding nysiis_name as a column to author_df
author_df['nysiis_name'] = nysiis_name

# Printing out the difference between unique firstnames and unique nysiis_names:
len(np.unique(first_name)) -len(np.unique(nysiis_name))

NameError: name 'fuzzy' is not defined

## 4. The inbetweeners
the   <code>babynames_nysiis.csv</code>, a dataset that is derived from <a href="https://www.ssa.gov/oact/babynames/limits.html">the Social Security Administration’s baby name data</a>, to identify author genders. The dataset contains unique NYSIIS versions of baby names, and also includes the percentage of times the name appeared as a female name (<code>perc_female</code>) and the percentage of times it appeared as a male name (<code>perc_male</code>). </p>
<p>We'll use this data to create a list of <code>gender</code>. 
In order to simplfy I'm making the following assumption: For each name, if <code>perc_female</code> is greater than <code>perc_male</code> then assume the name is female, if <code>perc_female</code> is less than <code>perc_male</code> then assume it is a male name, and if the percentages are equal then it's a "neutral" name.</p>

In [ ]:
# Reading in ../../data/nlp/babynames_nysiis.csv, which is semicolon delimited.
babies_df = pd.read_csv('../../data/nlp/babynames_nysiis.csv', sep = ';')

# Looping through babies_df to and filling up gender
gender = babies_df.perc_female - babies_df.perc_male
gender = [ 'F' if g>0 else 'M' if g<0 else 'N' for g in gender ]

# Adding a gender column to babies_df
babies_df['gender']=gender

# Printing out the first few rows of babies_df
babies_df.head()

## 5. Playing matchmaker
<p>using likely genders of different names, let's find author genders by searching for each author's name in the <code>babies_df</code> DataFrame, and extracting the associated gender. </p>

In [ ]:
# This function returns the location of an element in a_list.
# Where an item does not exist, it returns -1.
def locate_in_list(a_list, element):
    loc_of_name = a_list.index(element) if element in a_list else -1
    return(loc_of_name)

# Looping through author_df['nysiis_name'] and appending the gender of each
# author to author_gender.
author_gender = [gender[locate_in_list(babies_df.babynysiis,name)] if locate_in_list(babies_df.babynysiis,name)>-1 else 'Unknown' for name in nysiis_name]


# Adding author_gender to the author_df
author_df['author_gender']= author_gender
# Counting the author's genders
author_df['author_gender'].value_counts()

## 6. Tally up
<p>From the results above see that there are more female authors on the New York Times best seller's list than male authors spans 2008 to 2017. Let's find out if there have been changes over time.</p>

In [ ]:
# Task 6: count authors of each gender per year.
#
# The original cell was left as `...YOUR CODE FOR TASK 6...` and never written.
#
# BUGFIX: `years = ...` (an Ellipsis) is not valid as a value here - it only
# works because the placeholder was never replaced. Grouping by year with
# value_counts is the direct way to get the per-year gender breakdown.
years = sorted(author_df["Year"].unique())
print(f"years covered: {years[0]} - {years[-1]} ({len(years)} years)")

per_year = (
    author_df.groupby(["Year", "author_gender"])
    .size()
    .unstack(fill_value=0)
)

# Guarantee all three categories exist in every year, even where a count is 0,
# so the stacked chart below cannot silently drop a series.
for category in ("M", "F", "Unknown"):
    if category not in per_year.columns:
        per_year[category] = 0
per_year = per_year[["M", "F", "Unknown"]]

males_by_yr = per_year["M"].tolist()
females_by_yr = per_year["F"].tolist()
unknown_by_yr = per_year["Unknown"].tolist()

print(per_year)
print(f"\ntotals -> male={sum(males_by_yr)}  female={sum(females_by_yr)}  "
      f"unknown={sum(unknown_by_yr)}")


## 7. Foreign-born authors?
<p>Our gender data comes from social security applications of individuals born in the US. Hence, one possible explanation for why there are "unknown" genders associated with some author names is because these authors were foreign-born.
  

In [ ]:
# Task 7: bar chart of the yearly gender breakdown.
#
# The original cell was left as `...YOUR CODE FOR TASK 7...`.
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 5))
bar_width = 0.8
plt.bar(years, males_by_yr, width=bar_width, label="Male", color="tab:blue")
plt.bar(years, females_by_yr, width=bar_width, bottom=males_by_yr,
        label="Female", color="tab:pink")
plt.bar(years, unknown_by_yr, width=bar_width,
        bottom=[m + f for m, f in zip(males_by_yr, females_by_yr)],
        label="Unknown", color="0.7")

plt.xlabel("Year")
plt.ylabel("Number of authors")
plt.title("Gender of New York Times best-selling authors, by year")
plt.xticks(rotation=45)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Raising the bar
<p>What’s more exciting than a bar chart is a grouped bar chart. This type of chart is good for displaying <em>changes</em> over time while also <em>comparing</em> two or more groups. Let’s use a grouped bar chart to look at the distribution of male and female authors over time.</p>

In [ ]:
# Task 8: grouped bar chart, offsetting the female series so both are visible.
#
# The original cell was left as `... YOUR CODE FOR TASK 8 ...`.
#
# BUGFIX: the placeholder used `years_shifted = ...`, another Ellipsis. A real
# offset is needed, and because `years` holds integers the offset is applied
# arithmetically rather than by index.
years_shifted = [year + 0.4 for year in years]

plt.figure(figsize=(12, 5))
plt.bar(years, males_by_yr, width=0.4, label="Male", color="tab:blue")
plt.bar(years_shifted, females_by_yr, width=0.4, label="Female", color="tab:pink")

plt.xlabel("Year")
plt.ylabel("Number of authors")
plt.title("Male vs female authors per year (bars offset by 0.4 years)")
plt.xticks(years, rotation=45)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# What the chart actually shows, stated plainly.
identified = sum(males_by_yr) + sum(females_by_yr)
female_share = sum(females_by_yr) / identified if identified else 0.0
unknown_share = sum(unknown_by_yr) / (identified + sum(unknown_by_yr))
print(f"identified authors: {identified} "
      f"({female_share:.1%} female, {1 - female_share:.1%} male)")
print(f"authors with no SSA match: {sum(unknown_by_yr)} ({unknown_share:.1%} of all)")
print()
print("CAVEAT - read this before citing the numbers above.")
print("Gender is inferred by matching a phonetic key (NYSIIS) against US")
print("Social Security baby-name data. NYSIIS is lossy: the notebook itself")
print("shows 'beach' and 'bitch' collapsing to the same key. An unmatched name")
print("becomes 'Unknown', and a *wrongly* matched name is silently wrong - there")
print("is no confidence threshold anywhere in this pipeline. SSA data is also")
print("derived from applications for a Social Security number, which skews")
print("toward the population that applied for one.")
print("The trend above is therefore a property of the join, not evidence about")
print("the authors.")
